# 03. 0단계 점검 — 세탁만 그래프의 구조

유형 모델 상한선 실험은 **세탁 거래만 모은 그래프**(분할 끝까지 누적)를 입력으로 쓴다. 학습 전에 이 그래프가 어떻게 생겼는지 본다.
상한선 성능이 예상보다 높게 나올 때, 그게 구조를 배운 결과인지 쉬운 규칙 덕분인지 가를 근거가 여기서 나온다.

1. **이웃 종류** — 대상 엣지의 양끝 계좌에 붙은 다른 세탁 엣지가 없음 / UNMATCHED 만 / 패턴만 / 섞임.
   `given_kind` 에서 "isolated 면 거의 UNMATCHED" 처럼 한 클래스로 쏠리면 → 상한선의 UNMATCHED 성능은 쉬운 규칙. 이진 오탐이 섞이는 실제 조건(8단계)에서 크게 떨어질 것
2. **연결 요소** — 요소 하나 = 시도 하나인지, 여러 시도·UNMATCHED 가 엉켜 있는지, 거대 요소가 있는지
3. **계좌 공유** — 한 계좌가 여러 시도에 쓰이는지, 다른 유형 시도와 섞이는지

그래프 시점: 1 은 분할마다 그 분할 끝, 2·3 은 `CUTOFF`(기본 테스트 끝)

## 0. 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220)

sys.path.append(str(Path.cwd()))
import mc_lib as M

DATASET = "HI-Large"
TRAIN = ("2022-08-08", "2022-10-02")
VAL   = ("2022-10-03", "2022-10-16")
TEST  = ("2022-10-17", "2022-10-30")
CUTOFF = TEST[1]                      # 연결 요소·계좌 공유를 볼 시점 (이 날짜까지 누적)

windows = M.make_windows(TRAIN, VAL, TEST)
OUT = M.results_dir(DATASET, "03_graph_check")
lau = M.load_laundering(DATASET)
print(f"세탁 엣지 {len(lau):,}")

## 1. 이웃 종류

`by_class` : 분할 × 클래스 → 이웃 종류 비율(%), `other_med` = 양끝 계좌의 다른 세탁 엣지 수 중앙값

In [ ]:
nc = M.neighbor_context(lau, windows)
nc["by_class"]

`given_kind` : 분할 × 이웃 종류 → 클래스 분포(%), 행 합 100. 한 클래스로 쏠린 행이 쉬운 규칙

In [ ]:
nc["given_kind"]

## 2. 연결 요소 (방향 무시)

- `summary` : 요소 수, 요소 크기(엣지 수) 분위, 가장 큰 요소
- `kinds` : 요소 종류별 개수·엣지 비율 — `unmatched_only` / `single_attempt`(시도 1개만) / `single_attempt+unm` / `multi_attempt`
- `by_pattern` : 패턴별 시도 중 자기 요소에 다른 시도가 있는 비율, UNMATCHED 가 섞인 비율
- `by_class` : 클래스별 엣지가 어떤 종류 요소에 있나 (%)

In [ ]:
cc = M.components(lau, CUTOFF)
cc["summary"]

In [ ]:
cc["kinds"]

In [ ]:
cc["by_pattern"]

In [ ]:
cc["by_class"]

In [ ]:
# 큰 요소 상위 10개
cc["comp"].sort_values("n_edges", ascending=False).head(10)

## 3. 계좌 공유

- `accounts` : 패턴 계좌별 참여 시도 수 (1 = 시도 하나에만)
- `by_pattern` : 패턴별 시도 중 다른 시도와 계좌 공유 / 다른 유형 시도와 공유 / UNMATCHED 엣지가 있는 계좌 포함 비율(%)

In [ ]:
sa = M.shared_accounts(lau, CUTOFF)
sa["accounts"]

In [ ]:
sa["by_pattern"]

## 4. 저장

In [ ]:
for name, df in {"neighbor_by_class": nc["by_class"], "neighbor_given_kind": nc["given_kind"],
                 "components_summary": cc["summary"], "components_kinds": cc["kinds"],
                 "components_by_pattern": cc["by_pattern"], "components_by_class": cc["by_class"],
                 "shared_accounts": sa["accounts"], "shared_by_pattern": sa["by_pattern"]}.items():
    df.to_csv(OUT / f"{name}.csv", encoding="utf-8-sig")
sorted(p.name for p in OUT.iterdir())